Before you can reason about data structures, you need to understand where data actually lives. Every variable, object, and collection you create ends up as bytes in RAM. How those bytes are arranged — and how your program finds them — is what determines whether an operation takes one step or a million. This notebook builds that mental model from the ground up.

## What is RAM?

RAM (Random Access Memory) is your computer's working memory — the space where everything your program uses while it's running lives. Think of it as a very long row of small numbered slots, where each slot holds exactly one byte (8 bits).

When your program starts, the operating system gives it a chunk of RAM to work with. From that point on, every variable, every object, every data structure you create is just bytes occupying some of those slots.

Two key properties make RAM powerful:
- **Random access** — you can jump to any slot instantly, regardless of where you currently are. Slot 0 and slot 10,000,000 take the same time to reach.
- **Addresses** — every slot has a unique number (its address). Your program uses these addresses to find data.

## Bits, Bytes, and Addresses

The smallest unit of storage is a **bit** — a single 0 or 1. Eight bits form a **byte**, and a byte is the smallest unit RAM can address individually.

Each byte has a unique **memory address** — typically written in hexadecimal (e.g. `0x0000`, `0x0001`). When you store an integer, it occupies multiple consecutive bytes, all with sequential addresses.

![RAM address layout](https://raw.githubusercontent.com/schemabotview/dsa/main/img/ram-address-layout.svg)

Notice a few things:
- A 32-bit integer takes **4 bytes** (addresses `0x0000`–`0x0003`).
- A single character takes **1 byte** (`0x0004`).
- The values are stored in consecutive slots — they don't jump around.

This is why knowing the **size of a type** matters — it tells you how many bytes get reserved, and how far apart array elements are in memory.

### Python

`sys.getsizeof()` shows how many bytes Python uses to represent a value. `id()` returns the memory address of an object.

In [ ]:
import sys

x = 42
name = "Alice"
numbers = [1, 2, 3]

print(f"int    : {sys.getsizeof(x)} bytes  — address: {hex(id(x))}")
print(f"str    : {sys.getsizeof(name)} bytes  — address: {hex(id(name))}")
print(f"list   : {sys.getsizeof(numbers)} bytes  — address: {hex(id(numbers))}")

> **Note:** Python's integers are larger than a raw 4 bytes because Python wraps them in objects with extra metadata (type info, reference count). The raw data you care about for DSA purposes is the concept, not CPython's exact overhead.

### Kotlin

Kotlin (on the JVM) gives you both primitive and boxed types. Primitive `Int` compiles down to a Java `int` — 4 bytes on the stack. `Integer` (boxed) is an object on the heap.

```kotlin
fun main() {
    val x: Int = 42                      // primitive — 4 bytes on the stack
    val boxed: Int? = 42                 // boxed Integer object — lives on the heap

    println(System.identityHashCode(x))      // memory identity of the boxed form
    println(System.identityHashCode(boxed))

    // Int.SIZE_BYTES tells you the raw size of the primitive
    println("Int size: ${Int.SIZE_BYTES} bytes")     // 4
    println("Long size: ${Long.SIZE_BYTES} bytes")   // 8
    println("Char size: ${Char.SIZE_BYTES} bytes")   // 2 (UTF-16)
}
```

## The Stack and the Heap

Your program's memory is divided into two main regions with very different behaviors:

| | Stack | Heap |
|---|---|---|
| What lives here | Local variables, function arguments, return addresses | Objects, collections, dynamically sized data |
| Size | Fixed and small (usually ~1–8 MB) | Large (limited by available RAM) |
| Speed | Very fast — just move a pointer | Slower — allocator must find free space |
| Lifetime | Auto-freed when function returns | Freed by garbage collector (or manually in C/C++) |
| Structure | Ordered — last in, first out | Unordered — objects scatter wherever space exists |

![Stack vs Heap](https://raw.githubusercontent.com/schemabotview/dsa/main/img/stack-vs-heap.svg)

When `add()` is called from `main()`, a new **stack frame** is pushed — it holds the function's local variables and parameters. When `add()` returns, the frame is popped and that memory is instantly reclaimed. No cleanup needed.

The heap is messier but far more flexible. Objects live there until nothing references them anymore, at which point the garbage collector reclaims the space.

### Python

In [ ]:
# Every time add() is called, a new stack frame is created.
# Python's call stack is visible via the traceback.

def add(a, b):
    result = a + b       # 'result', 'a', 'b' live in this frame
    return result        # frame is destroyed here

def main():
    x = 10               # lives in main's frame
    y = 20
    total = add(x, y)    # new frame pushed for add()
    print(total)         # add()'s frame is already gone

main()

### Kotlin

```kotlin
fun add(a: Int, b: Int): Int {
    val result = a + b   // a, b, result live in this stack frame
    return result        // frame is popped, memory freed automatically
}

fun main() {
    val x = 10           // stack
    val y = 20           // stack
    val total = add(x, y)
    println(total)
}
```

## Value Types vs Reference Types

This is one of the most important distinctions in programming — and it directly affects how data structures behave.

- **Value type** — the variable *contains* the value directly. Copying the variable copies the value.
- **Reference type** — the variable contains an *address* pointing to an object on the heap. Copying the variable copies the address — both variables now point to the same object.

![Value vs Reference](https://raw.githubusercontent.com/schemabotview/dsa/main/img/value-vs-reference.svg)

### Python

In [ ]:
# In Python, everything is an object on the heap.
# Variables are always references (addresses).

# Integers — Python caches small ints, so same value = same address
a = 42
b = a
print(f"a is b: {a is b}")  # True — same cached object
print(f"id(a): {hex(id(a))}, id(b): {hex(id(b))}")

# Lists — mutable objects on the heap
list_a = [1, 2, 3]
list_b = list_a          # copies the reference, not the list
list_b.append(4)

print(f"list_a: {list_a}")  # [1, 2, 3, 4] — both point to the same list!
print(f"list_b: {list_b}")  # [1, 2, 3, 4]
print(f"Same object: {list_a is list_b}")

### Kotlin

```kotlin
fun main() {
    // Value type — Int is a primitive on the stack
    var a = 42
    var b = a      // copy of the value
    b = 99
    println(a)     // 42 — unchanged, b got its own copy

    // Reference type — MutableList lives on the heap
    val listA = mutableListOf(1, 2, 3)
    val listB = listA     // copy of the reference (address)
    listB.add(4)

    println(listA)        // [1, 2, 3, 4] — same object!
    println(listB)        // [1, 2, 3, 4]
    println(listA === listB)  // true — same reference
}
```

## Why This Matters for Data Structures

Everything you will learn in this course connects back to memory layout:

**Arrays** — elements sit in consecutive memory slots. That's why index access is O(1): the address of element `i` is just `base_address + i × element_size`. No searching needed.

**Linked lists** — each node is a separate heap object with a pointer (reference) to the next. Nodes can be anywhere in memory. That's why index access is O(n): you must follow each pointer in sequence.

**Hash maps** — use an array under the hood, but compute a slot address from the key. O(1) average, because it's just arithmetic to find the right slot.

**Trees and graphs** — nodes on the heap, connected by references. Traversal cost depends on how many references you follow.

The pattern is always the same: **contiguous memory = fast index access, pointer-linked memory = flexible structure but slower traversal**.

| Data Structure | Memory Layout | Index | Insert | Why |
|---|---|---|---|---|
| Array | Contiguous | O(1) | O(n) | Direct address math / shifting elements |
| Linked List | Scattered (pointers) | O(n) | O(1) | Must traverse / just rewire a pointer |
| Hash Map | Contiguous (array + hash) | O(1) avg | O(1) avg | Hash gives slot directly |

## Key Takeaways

- RAM is a long row of numbered byte-slots. Every value your program uses occupies some of those slots.
- Each byte has a unique **memory address**. Your program finds data by its address.
- The **stack** holds short-lived locals — fast, auto-managed, small. The **heap** holds objects — flexible, garbage-collected, large.
- **Value types** store data directly in the variable. **Reference types** store an address pointing to the real data on the heap. Copying a reference copies the address, not the object.
- Data structure performance is a direct consequence of memory layout: contiguous = fast random access, pointer-linked = flexible but sequential traversal.